In [ ]:
# Cell 1: Import dan konfigurasi awal (versi untuk Kaggle)
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG16
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau, EarlyStopping

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score, precision_recall_fscore_support, roc_auc_score, classification_report

# konfigurasi dasar
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

# path dataset di Kaggle
# struktur dataset: /kaggle/input/cell-images-for-detecting-malaria/cell_images/Parasitized/*.png
#                   /kaggle/input/cell-images-for-detecting-malaria/cell_images/Uninfected/*.png
dataset_dir = "/kaggle/input/cell-images-for-detecting-malaria/cell_images"

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 10
LR = 1e-4

# cek GPU
print("TensorFlow version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))
print("Dataset directory:", os.listdir(dataset_dir))

In [ ]:
# Cell 2: Membuat dataframe dari folder dataset (versi untuk Kaggle)
import os
import pandas as pd

filepaths = []
labels = []

# Loop untuk membuat dataframe
for class_name in ["Parasitized", "Uninfected"]:
    class_folder = os.path.join(dataset_dir, class_name)
    # pastikan folder ada
    if not os.path.exists(class_folder):
        print(f"Folder tidak ditemukan: {class_folder}")
        continue

    for fname in os.listdir(class_folder):
        fpath = os.path.join(class_folder, fname)
        # pastikan benar-benar file gambar
        if os.path.isfile(fpath) and fname.lower().endswith(('.jpg', '.jpeg', '.png')):
            filepaths.append(fpath)
            labels.append(class_name)

# buat dataframe
df = pd.DataFrame({"filepath": filepaths, "label": labels})
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)

print("Jumlah total gambar:", len(df))
print(df['label'].value_counts())
print("\nContoh data:")
print(df.head())

In [ ]:
# Cell 3: Split data (versi untuk Kaggle)
from sklearn.model_selection import train_test_split

# Split data dengan stratifikasi agar seimbang
train_df, test_df = train_test_split(
    df, test_size=0.2, stratify=df['label'], random_state=SEED
)

train_df, val_df = train_test_split(
    train_df, test_size=0.2, stratify=train_df['label'], random_state=SEED
)

print(f"Train: {len(train_df)}")
print(f"Validation: {len(val_df)}")
print(f"Test: {len(test_df)}")

# tampilkan distribusi label untuk memastikan pembagian seimbang
print("\nDistribusi label:")
print("Train:\n", train_df['label'].value_counts())
print("\nValidation:\n", val_df['label'].value_counts())
print("\nTest:\n", test_df['label'].value_counts())

# tampilkan contoh beberapa baris
print("\nContoh data train:")
print(train_df.head())

In [ ]:
# Cell 4: ImageDataGenerator (augmentasi dan normalisasi) - versi untuk Kaggle
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Augmentasi data training
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=25,
    zoom_range=[0.8, 1.0],
    horizontal_flip=True,
    fill_mode='nearest'
)

# Normalisasi untuk validasi dan testing (tanpa augmentasi)
val_test_datagen = ImageDataGenerator(rescale=1./255)

# Generator training
train_gen = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    x_col='filepath',
    y_col='label',
    target_size=IMG_SIZE,
    class_mode='binary',
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED,
    dtype='float32'
)

# Generator validation
val_gen = val_test_datagen.flow_from_dataframe(
    dataframe=val_df,
    x_col='filepath',
    y_col='label',
    target_size=IMG_SIZE,
    class_mode='binary',
    batch_size=BATCH_SIZE,
    shuffle=False,
    dtype='float32'
)

# Generator testing
test_gen = val_test_datagen.flow_from_dataframe(
    dataframe=test_df,
    x_col='filepath',
    y_col='label',
    target_size=IMG_SIZE,
    class_mode='binary',
    batch_size=BATCH_SIZE,
    shuffle=False,
    dtype='float32'
)

print("\nClass indices mapping:")
print(train_gen.class_indices)

print(f"\nTotal batch (train): {len(train_gen)}")
print(f"Total batch (val): {len(val_gen)}")
print(f"Total batch (test): {len(test_gen)}")

In [ ]:
# Cell 5: Model VGG16 (versi untuk Kaggle)
from tensorflow.keras.applications import VGG16
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

# Gunakan VGG16 pretrained dari ImageNet tanpa fully connected layer (include_top=False)
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# Freeze semua layer convolutional agar tidak ikut dilatih
for layer in base_model.layers:
    layer.trainable = False

# Tambahkan layer kustom untuk klasifikasi malaria
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.4)(x)
outputs = Dense(1, activation='sigmoid')(x)

# Bentuk model akhir
model = Model(inputs=base_model.input, outputs=outputs)

# Kompilasi model
model.compile(
    optimizer=Adam(learning_rate=LR),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Tampilkan ringkasan model dan status GPU
print("Model siap untuk training di Kaggle environment.")
print("GPU Available:", tf.config.list_physical_devices('GPU'))
model.summary()

In [ ]:
# Cell 6: Training dan waktu eksekusi (versi untuk Kaggle)
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau, EarlyStopping
import time

# Simpan model terbaik ke folder /kaggle/working agar bisa diunduh dari tab "Output"
checkpoint_path = "/kaggle/working/best_vgg16_malaria.h5"
checkpoint_cb = ModelCheckpoint(
    filepath=checkpoint_path,
    save_best_only=True,
    monitor="val_accuracy",
    mode="max",
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    verbose=1
)

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

print("Mulai training model...")
start_time = time.time()

history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS,
    callbacks=[checkpoint_cb, reduce_lr, early_stop],
    verbose=1
)

end_time = time.time()
print(f"\nWaktu training total: {end_time - start_time:.2f} detik")
print(f"Model terbaik disimpan di: {checkpoint_path}")

In [ ]:
# Cell 7: Evaluasi pada data test

from sklearn.metrics import confusion_matrix, accuracy_score, precision_recall_fscore_support, roc_auc_score
import seaborn as sns
import matplotlib.pyplot as plt

# Prediksi probabilitas dan label
y_prob = model.predict(test_gen, verbose=1)
y_pred = (y_prob > 0.5).astype(int).flatten()
y_true = test_gen.classes

# Hitung metrik evaluasi
cm = confusion_matrix(y_true, y_pred)
accuracy = accuracy_score(y_true, y_pred)
precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='binary')
auc = roc_auc_score(y_true, y_prob)

print("Confusion Matrix:\n", cm)
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"AUC/ROC: {auc:.4f}")

# Visualisasi confusion matrix
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
# Cell 8: Plot Confusion Matrix

import numpy as np
import matplotlib.pyplot as plt

def plot_confusion_matrix(cm, classes):
    fig, ax = plt.subplots(figsize=(5,4))
    im = ax.imshow(cm, cmap='Blues')
    ax.figure.colorbar(im, ax=ax)
    ax.set_xticks(np.arange(len(classes)))
    ax.set_yticks(np.arange(len(classes)))
    ax.set_xticklabels(classes)
    ax.set_yticklabels(classes)
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title('Confusion Matrix')

    thresh = cm.max() / 2
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, cm[i, j], ha='center', va='center',
                    color='white' if cm[i, j] > thresh else 'black')
    plt.show()

plot_confusion_matrix(cm, ["Parasitized", "Uninfected"])

In [ ]:
# Cell 9: Classification Report

from sklearn.metrics import classification_report

print(classification_report(
    y_true,
    y_pred,
    target_names=["Parasitized", "Uninfected"]
))

In [ ]:
# ============================
# # Cell 10 — PLOT LOSS & ACCURACY
# ============================
plt.figure(figsize=(14,5))
plt.subplot(1,2,1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.title('Accuracy')
plt.legend()
plt.subplot(1,2,2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Loss')
plt.legend()
plt.show()

In [ ]:
# Save as H5
model.save("/kaggle/working/malaria_model.h5")

# Save as native Keras format
model.save("/kaggle/working/malaria_model.keras")

# Save as TensorFlow SavedModel folder (for export/zip)
model.export("/kaggle/working/malaria_model_folder")

# Zip the folder
!zip -r /kaggle/working/malaria_model.zip /kaggle/working/malaria_model_folder
